# Citadel T1D + PRE50M — ONE-SHOT TPU RUN (final pre-50M diagnostic)

Preregistered: `docs/citadel/experiments/T1D/PLAN.md` + `PRE50M_ADDENDUM.md` +
`SELF_KNOWLEDGE_AMENDMENT.md` (Arm F). Six frozen arms — A flat / B curriculum /
C teacher / D scale / E repr-diagnostic / F self-knowledge — then the PRE50M
systems certification, in ONE orchestrated run.

## Operator workflow
1. Select the TPU runtime.
2. Run CELL 0 (bootstrap). It prints the SHAs and stops on any mismatch.
3. Run CELL 1 (RUN EVERYTHING) and wait (~70-110 min).
4. The result bundle downloads automatically:
   `CITADEL_T1D_RESULTS.zip` on success, `CITADEL_T1D_FAILURE.zip` on any
   failure (with the exact phase, gate list, and traceback inside).
No other cells. No manual patching. No log screenshots.

In [ ]:
# CELL 0 — bootstrap: fresh Citadel checkout + pinned read-only Cymek runtime
import os, subprocess, sys
repo = '/content/An-Ra-colab'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git','clone','--depth','50','-b','citadel','https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git',repo], check=True)
else:
    subprocess.run(['git','-C',repo,'fetch','origin','citadel','--depth','50'], check=True)
    subprocess.run(['git','-C',repo,'checkout','citadel'], check=True)
    subprocess.run(['git','-C',repo,'reset','--hard','origin/citadel'], check=True)
%cd /content/An-Ra-colab
os.environ['CITADEL_ROOT'] = repo
os.environ.setdefault('PJRT_DEVICE', 'TPU')
os.environ['CITADEL_PLATFORM'] = 'colab'
if repo not in sys.path: sys.path.insert(0, repo)
from citadel_tpu import runtime_bootstrap as rb
rt_root, rt_sha = rb.ensure_cymek_runtime()
print('CITADEL_SHA=' + str(rb.citadel_sha()))
print('CYMEK_RUNTIME_SHA=' + str(rt_sha))
print('EXPECTED_CYMEK_SHA=28bf57a0d299a2c13a99fe0046616c00a1b8530c')
if rt_sha != '28bf57a0d299a2c13a99fe0046616c00a1b8530c':
    raise RuntimeError('CYMEK PIN MISMATCH: runtime ' + rt_sha + ' != pinned expected - STOP')
SESSION = 'docs/citadel/tpu_receipts/t1d_session'
print('SESSION_DIR=' + SESSION)

In [ ]:
# CELL 1 — RUN EVERYTHING: preflight -> canary -> data -> arms A-F -> PRE50M -> bundle
# Stale-module hardening: reload execution modules right before the run so this
# cell always executes the freshly pulled code, never a stale in-memory copy.
import importlib
from citadel_tpu import t1d_one_shot as _oshot
from citadel_tpu import pre50m as _p50
from citadel_tpu import t1d_run as _t1d_module
import citadel_tpu
_oshot = importlib.reload(_oshot)
_p50 = importlib.reload(_p50)
t1d = importlib.reload(_t1d_module)
print('one-shot orchestrator:', _oshot.ORCHESTRATOR_VERSION, '| phases:', len(_oshot.PHASE_ORDER))
session = _oshot.run_all(SESSION)
print('ONE-SHOT STATUS:', session['status'])
print('phases:', session['phases'])
if 'labels' in session: print('T1D labels:', session['labels'])
from google.colab import files
if session['status'] == 'COMPLETE':
    print('verifying bundle before download...')
    print(t1d.verify_bundle(SESSION)['status'])
    files.download(SESSION + '/CITADEL_T1D_RESULTS.zip')
    print('RESULT BUNDLE downloaded')
else:
    print('FAILURE BUNDLE:', session.get('failure_bundle'))
    files.download(session.get('failure_bundle') or (SESSION + '/CITADEL_T1D_FAILURE.zip'))
    print('FAILURE BUNDLE downloaded - send it back; do not screenshot anything')